# AAV2 sélectivité — filtrage sur la profondeur du *readout*

Port direct de `AAV5/selectivity/sorting/AAV5_SEL_potts_readout_depth.ipynb` — **la même méthode**, appliquée à AAV2 (le filtre à utiliser était explicitement demandé comme "analogue au meilleur filtre trouvé avec AAV5"). Même modèle de Potts (F additif + J pairwise, 8541 features : single-site one-hot + pairwise outer-product + biais), même loss (moindres carrés gaussiens, ridge L2), même solveur (`RegressionV1.fit_weights_potts_from_data`), mêmes poids d'observation inverse-variance `w = 1/(1/n_num + 1/n_den)` (`eps=0.5`, convention permanente du projet).

## Le critère

$$y_i \;=\; \log_2\!\frac{\text{compte\_organoïde\_adn}_i}{\text{compte\_virus}_i}$$

Ni le numérateur ni le dénominateur d'un `y_i` brut ne sont filtrés par construction — sur AAV5 c'était la cause du plafond bas de la CV (cible dominée par le bruit d'échantillonnage Poisson/multinomial à faible comptage). La correction retenue là-bas, et celle qu'on reprend ici telle quelle : ne garder que les variants où

$$\text{compte\_organoïde\_adn} \ge T \quad\textbf{et}\quad \text{compte\_virus} \ge T$$

— une exigence de **correspondance** entre le numérateur ET le dénominateur (org2 ET org3 doivent chacun avoir un signal) combinée à un **minimum de comptage** (`T`, pas juste `>0`) — balayée de 0 à 100. Cibles : `sel_org2` et `sel_org3` (org1 exclu du fit, comme sur AAV5, gardé seulement dans le plafond de reproductibilité §1 — à confirmer que c'est aussi l'outlier ici, pas supposé d'office).

## Différence avec le port AAV5 (dataset)

AAV5 partait de `AAV5_organoides_sorted.csv` (CSV déjà filtré sur l'axe viabilité, TOUTES les colonnes gardées). Pour AAV2, `AAV2_organoides_sorted.csv` (produit par `AAV2_viab_sorting.ipynb` §8) ne garde que `sequence`/`compte_plasmide`/`compte_virus`/`log2_enrichissement_virus_sur_plasmide` (`usecols` restreint à dessein, organoïde/noyaux hors-scope viabilité à l'époque) — il manque les colonnes organoïde nécessaires ici. Ce notebook charge donc le CSV BRUT `AAV2_organoides.csv` (34 colonnes) directement, et redérive en ligne un filtre qualité-viab minimal comme population de base, avant d'y empiler le seuil de profondeur du readout. Pas de retrait type 7m8 (hamming) : `AAV2_viab_sorting.ipynb` §3 n'a trouvé aucun spike-in côté plasmide analogue au 7m8 d'AAV5.

**⚠ Correction (2026-09-16) : le cap `ratio_virus/plasmide ≤ 100` (hérité de `AAV2_viab_sorting.ipynb` §8) a été retiré.** Un cap sur le RATIO équivaut par construction à tronquer tout `log2 enrichment` réel au-delà de `log2(100)=6.64` — vérifié sur les données brutes : ça rayait 100% des variants à `y>6.64` et déjà 75% de ceux à `y>6`. Pas un filtre de bruit, une amputation de la queue haute. Seul filtre qualité restant : `compte_plasmide ≥ 1`. **Toute la sweep de ce notebook (sections 1 à 8, y compris la décision `T_CHOSEN=5` en section 9) a été calculée sur l'ANCIENNE population (avec le cap) — à rejouer intégralement avant de faire confiance aux chiffres affichés.**

**⚠ 2026-09-18 : régression de Potts reconstruite avec le nouveau solveur par défaut du projet** (`RegressionV1.fit_weights_potts_from_data_matrixfree` -- matrix-free, plus de matrice de design dense matérialisée ; validé pour reproduire le solve classique quasi exactement, `r(F)`/`r(J) > 0.999999` sur données AAV2 réelles -- cf. `viability/AAV2_potts_ridge_matrixfree_validation.ipynb`). Même signature d'appel/retour que l'ancienne `fit_weights_potts_from_data` (`rank` vaut toujours `None` ici -- pas de diagnostic SVD avec un solveur itératif). L'ancienne version de ce notebook (solveur dense/SVD) est archivée dans `obsolete_dense_matrix_potts_regression/`. **Sorties de cellules effacées** (méthode de fit changée, anciens chiffres plus valides) -- à ré-exécuter. Notebook préparé mais **jamais exécuté par Claude** (`feedback_user_runs_notebooks`).

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
LIB = _root / "lib"

import RegressionV1 as R
from analysisV1 import AA_LABELS, pearson, topk_recovery
assert "Modelization_V2" in R.__file__, R.__file__
print(R.message)

CSV = Path("AAV2_organoides.csv")
if not CSV.exists():
    CSV = _root / "notebooks/notebooks/AAVs dataset/AAV2/AAV2_organoides.csv"
assert CSV.exists(), CSV

viab_col = "log2_enrichissement_virus_sur_plasmide"
sel = {1: "log2_enrichissement_organoide_1_adn_sur_virus",
       2: "log2_enrichissement_organoide_2_adn_sur_virus",
       3: "log2_enrichissement_organoide_3_adn_sur_virus"}
cnt = {i: f"compte_organoide_{i}_adn" for i in (1, 2, 3)}
use = ["sequence", "compte_plasmide", "compte_virus", viab_col, *cnt.values(), *sel.values()]
df = pd.read_csv(CSV, usecols=use,
                  dtype={c: "float32" for c in use if c != "sequence"} | {"sequence": "string"})
df["sequence"] = df["sequence"].astype("string")
print(df.shape)

L, A = 7, 20
lut = np.zeros(256, np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i
seq_matrix = lut[np.frombuffer("".join(df["sequence"]).encode("ascii"), np.uint8)].reshape(len(df), L)

plasmid = df["compte_plasmide"].to_numpy(np.float64)
virus   = df["compte_virus"].to_numpy(np.float64)

# PAS de cap sur le ratio virus/plasmide (retire le 2026-09-16, cf. markdown en tete de notebook) --
# un cap ratio<=R equivaut par construction a tronquer tout log2 enrichment reel au-dela de
# log2(R) : a R=100 ca rayait 100% des variants a y>6.64 et deja 75% de ceux a y>6 -- verifie
# directement sur les donnees, pas un filtre de bruit. Seul filtre qualite restant :
# compte_plasmide >= PLASMID_MIN (exclut juste les lignes non-finies).
PLASMID_MIN = 1
viab_keep = (plasmid >= PLASMID_MIN)
print(f"filtre qualite-viab (plasmid>={PLASMID_MIN}, pas de cap ratio) : "
      f"{int(viab_keep.sum()):,}/{len(df):,} conserves ({viab_keep.mean():.1%})")


def score_FJ(seq, F, J):
    F, J = np.asarray(F), np.asarray(J)
    Fp = F[seq, np.arange(L)].sum(axis=1).astype(np.float64)
    Jp = np.zeros(len(seq), np.float64)
    for i in range(L):
        for j in range(i + 1, L):
            Jp += J[i, j, seq[:, i], seq[:, j]]
    return Fp + Jp


def obs_weight(num, den):
    return 1.0 / (1.0 / (num + 0.5) + 1.0 / (den + 0.5))


def spearman(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return pearson(np.argsort(np.argsort(a)), np.argsort(np.argsort(b)))


THRS       = np.array([0, 5, 10, 20, 30, 50, 100])
T_CHOSEN   = 5                           # decision du 2026-09-16 (r(F2,F3) plus eleve a T=5 qu'a
                                          # T=20) -- MESUREE SUR L'ANCIENNE POPULATION (ratio<=100),
                                          # a reconfirmer une fois ce notebook relance avec le
                                          # nouveau filtre (viab_keep change, donc la sweep ci-dessous
                                          # doit etre rejouee avant de faire confiance a T_CHOSEN=5)
# N_FIT retire (2026-09-18) -- plus de cap sur la population de fit, le solveur
# matrix-free n'a plus le plafond memoire de l'ancien solveur dense/SVD (~90-150k lignes)
LAMBDAS_CV = np.logspace(-4, 8, 21)      # grille CV elargie : 12 ordres de grandeur
KF_CV      = 3
print("seq_matrix:", seq_matrix.shape, "| THRS =", THRS, "| T_CHOSEN =", T_CHOSEN)
print(f"LAMBDAS_CV: {LAMBDAS_CV[0]:.0e} .. {LAMBDAS_CV[-1]:.0e}  ({len(LAMBDAS_CV)} points, {KF_CV}-fold)")

## 1. Plafond de reproductibilité vs seuil `T`

`r(y_org_i, y_org_j)` sur les variants où **les deux** réplicats passent le filtre `compte_organoïde ≥ T` ET `compte_virus ≥ T` (en plus du filtre qualité-viab de base). C'est le maximum qu'un fit peut viser à chaque `T`.

In [ ]:
pairs = [(2, 3), (1, 2), (1, 3)]
rows = []
for T in THRS:
    for (a, b) in pairs:
        ya, yb = df[sel[a]].to_numpy(np.float64), df[sel[b]].to_numpy(np.float64)
        m = (viab_keep & np.isfinite(ya) & np.isfinite(yb)
             & (df[cnt[a]].to_numpy() >= T) & (df[cnt[b]].to_numpy() >= T) & (virus >= T))
        rows.append({"T": int(T), "paire": f"org{a}-org{b}", "n": int(m.sum()),
                     "r(y,y)": round(float(np.corrcoef(ya[m], yb[m])[0, 1]), 3) if m.sum() > 50 else np.nan})
ceil = pd.DataFrame(rows)
ceil_wide = ceil.pivot(index="T", columns="paire", values="r(y,y)")
n_wide = ceil.pivot(index="T", columns="paire", values="n")
display(ceil_wide)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for p in ["org2-org3", "org1-org2", "org1-org3"]:
    ax1.plot(ceil_wide.index, ceil_wide[p], "o-", label=p)
ax1.set(xlabel="seuil T (compte organoïde ET virus ≥ T)", ylabel="r(y_i, y_j)",
        title="AAV2 -- Plafond de reproductibilité inter-réplicat")
ax1.legend(); ax1.grid(alpha=0.3)
for p in ["org2-org3", "org1-org2", "org1-org3"]:
    ax2.plot(n_wide.index, n_wide[p], "s-", label=p)
ax2.set(xlabel="seuil T", ylabel="n variants communs", yscale="log",
        title="Taille du jeu commun"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Balayage du seuil — fit ridge λ = 0 (OLS minimum-norm)

Pour chaque `T` et chaque réplicat (org2, org3) : masque `viab_keep & finite(y) & compte_organoïde ≥ T & compte_virus ≥ T`, split 50/50 (`random_state=0`), fit sur ≤ `N_FIT` lignes du train, held-out sur le test. Puis :
- **inter-réplicat** : le F/J de org2 score les lignes de org3 *hors du train de org2*, corrélé à `y_org3` (et symétrique).
- **accord des poids** : `r(F_org2, F_org3)`, `r(J_org2, J_org3)` hors-diagonale.

In [ ]:
def masks_at(T):
    out = {}
    for i in (2, 3):
        yi = df[sel[i]].to_numpy(np.float64)
        m = viab_keep & np.isfinite(yi) & (df[cnt[i]].to_numpy() >= T) & (virus >= T)
        out[i] = np.flatnonzero(m)
    return out

sweep, W = [], {}
for T in THRS:
    mi = masks_at(T)
    fitted = {}
    for i in (2, 3):
        idx = mi[i]
        yi = df[sel[i]].to_numpy(np.float64)
        wi = obs_weight(df[cnt[i]].to_numpy(np.float64), virus)
        tr, te = train_test_split(idx, test_size=0.5, random_state=0)
        tr_fit = tr  # plus de cap N_FIT (2026-09-18) -- solveur matrix-free, pas de plafond memoire
        F, J, rank, info = R.fit_weights_potts_from_data_matrixfree(
            seq_matrix[tr_fit], yi[tr_fit], sample_weight=wi[tr_fit], verbose=False, lam=0.0)
        F, J = np.asarray(F), np.asarray(J)
        r_self = pearson(yi[te], score_FJ(seq_matrix[te], F, J))
        fitted[i] = dict(F=F, J=J, tr_fit=set(tr_fit.tolist()), te=te, rank=rank, r_self=r_self)
        W[(T, i)] = (F, J)

    cross = {}
    for (i, j) in [(2, 3), (3, 2)]:
        yj = df[sel[j]].to_numpy(np.float64)
        ev = np.array([k for k in mi[j] if k not in fitted[i]["tr_fit"]])
        cross[(i, j)] = pearson(yj[ev], score_FJ(seq_matrix[ev], fitted[i]["F"], fitted[i]["J"])) if ev.size > 50 else np.nan

    od = ~np.eye(L, dtype=bool)
    sweep.append({
        "T": int(T),
        "n_org2": len(mi[2]), "n_org3": len(mi[3]),
        "rank_org2": fitted[2]["rank"], "rank_org3": fitted[3]["rank"],
        "r_self_org2": round(fitted[2]["r_self"], 3), "r_self_org3": round(fitted[3]["r_self"], 3),
        "r_cross_2to3": round(cross[(2, 3)], 3), "r_cross_3to2": round(cross[(3, 2)], 3),
        "r(F2,F3)": round(pearson(fitted[2]["F"].ravel(), fitted[3]["F"].ravel()), 3),
        "r(J2,J3)_offdiag": round(pearson(fitted[2]["J"][od].ravel(), fitted[3]["J"][od].ravel()), 3),
    })
    s = sweep[-1]
    print(f"T={T:>3}  n(org2/org3)={s['n_org2']:>7,}/{s['n_org3']:>7,}  "
          f"r_self={s['r_self_org2']:+.3f}/{s['r_self_org3']:+.3f}  "
          f"r_cross={s['r_cross_2to3']:+.3f}/{s['r_cross_3to2']:+.3f}  "
          f"r(F2,F3)={s['r(F2,F3)']:+.3f}  r(J2,J3)={s['r(J2,J3)_offdiag']:+.3f}")

sweep = pd.DataFrame(sweep).set_index("T")
sweep

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(ceil_wide.index, ceil_wide["org2-org3"], "k--o", lw=1.2, label="plafond r(y2,y3)")
ax.plot(sweep.index, sweep[["r_self_org2", "r_self_org3"]].mean(axis=1), "o-", color="crimson",
        label="held-out même réplicat (moy org2/org3)")
ax.plot(sweep.index, sweep[["r_cross_2to3", "r_cross_3to2"]].mean(axis=1), "s-", color="steelblue",
        label="held-out inter-réplicat (moy)")
ax.plot(sweep.index, sweep["r(F2,F3)"], "^-", color="green", label="accord des poids r(F2,F3)")
ax.plot(sweep.index, sweep["r(J2,J3)_offdiag"], "v-", color="olive", label="accord des poids r(J2,J3)")
ax.set(xlabel="seuil T (compte organoïde ET virus ≥ T)", ylabel="Pearson r",
       title="AAV2 -- Effet du filtrage sur la profondeur du readout -- fit ridge λ=0")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. La CV sur λ se comporte-t-elle mieux ?

CV `KF_CV`-fold sur **org2** à `T ∈ {0, T_CHOSEN, 50}`, avec la **grille λ élargie** `np.logspace(-4, 8, 21)`. Même question que sur AAV5 : la courbe CV MSE développe-t-elle un minimum **intérieur** propre quand la cible cesse d'être du bruit ?

In [ ]:
y2 = df[sel[2]].to_numpy(np.float64)
w2 = obs_weight(df[cnt[2]].to_numpy(np.float64), virus)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
cv_summary = []
for ax, T in zip(axes, [0, T_CHOSEN, 50]):
    idx = masks_at(T)[2]
    tr, te = train_test_split(idx, test_size=0.5, random_state=0)
    tr_fit = tr  # plus de cap N_FIT (2026-09-18) -- solveur matrix-free, pas de plafond memoire
    print(f"[§3] CV org2 T={T}  (n_fit={len(tr_fit):,}, {len(LAMBDAS_CV)} λ × {KF_CV} folds)...")
    F, J, rank, info = R.fit_weights_potts_from_data_matrixfree(
        seq_matrix[tr_fit], y2[tr_fit], sample_weight=w2[tr_fit],
        lambdas_grid=LAMBDAS_CV, k_folds=KF_CV, seed=0, verbose=True, lam=None)
    r_cv = pearson(y2[te], score_FJ(seq_matrix[te], np.asarray(F), np.asarray(J)))
    cv = np.asarray(info["cv_mse"]); lam = info["lam"]
    var_y = float(np.var(y2[tr_fit]))
    interior = LAMBDAS_CV[0] < lam < LAMBDAS_CV[-1]
    ax.plot(LAMBDAS_CV, cv, "o-", ms=3)
    ax.axhline(var_y, color="grey", ls=":", label=f"Var(y)={var_y:.2f}")
    ax.axvline(lam, color="crimson", ls="--", label=f"λ*={lam:.2g}")
    ax.set(xscale="log", xlabel="λ", ylabel="CV MSE",
           title=f"org2, T={T}  (λ* {'INTÉRIEUR' if interior else 'au bord'}, held-out r={r_cv:+.3f})")
    ax.legend(fontsize=8)
    cv_summary.append({"T": T, "lambda_CV": float(f"{lam:.4g}"), "interieur": interior,
                       "r_test_CV": round(r_cv, 3),
                       "r_test_lam0": round(float(sweep.loc[T, "r_self_org2"]), 3) if T in sweep.index else np.nan})
plt.tight_layout(); plt.show()
pd.DataFrame(cv_summary).set_index("T")

## 4. Poids F / J — cible bruitée (T=0) vs cible filtrée (T croissant)

org2, fit ridge λ=0, une ligne par seuil de `THRS`. Si le filtrage marche, la structure de F/J doit devenir plus nette et mieux reproduite par org3 (colonne de droite : F org2 vs F org3, corrélation) à mesure que `T` augmente.

In [ ]:
fig, axes = plt.subplots(7, 3, figsize=(16, 28))
for row, T in enumerate(THRS):
    F, J = W[(T, 2)]
    vmax = np.abs(F).max() or 1.0
    im = axes[row, 0].imshow(F, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[row, 0].set(title=f"F — org2, T={T}", xlabel="position", ylabel="acide aminé")
    axes[row, 0].set_xticks(range(L)); axes[row, 0].set_xticklabels(range(1, L + 1))
    axes[row, 0].set_yticks(range(A)); axes[row, 0].set_yticklabels(AA_LABELS, fontsize=6)
    fig.colorbar(im, ax=axes[row, 0], fraction=0.046)

    Jabs = np.abs(J).mean(axis=(2, 3)); off = Jabs.copy(); np.fill_diagonal(off, np.nan)
    im = axes[row, 1].imshow(off, cmap="viridis")
    axes[row, 1].set(title=f"mean|J_ij| — org2, T={T}  (max={np.nanmax(off):.3f})",
                     xlabel="position j", ylabel="position i")
    fig.colorbar(im, ax=axes[row, 1], fraction=0.046)

    F3, J3 = W[(T, 3)]
    axes[row, 2].scatter(F.ravel(), F3.ravel(), s=4, alpha=0.4)
    lim = max(np.abs(F).max(), np.abs(F3).max())
    axes[row, 2].plot([-lim, lim], [-lim, lim], "k--", lw=0.6)
    axes[row, 2].set(title=f"F : org2 vs org3, T={T}  (r={pearson(F.ravel(), F3.ravel()):+.3f})",
                     xlabel="F org2", ylabel="F org3")
plt.tight_layout(); plt.show()

## 5. Coût du filtrage en diversité

Combien de variants on garde selon `T` — et surtout combien ont **au moins un** des 3 réplicats qui passe le filtre (l'union, ce qu'on peut espérer scorer si on mutualise).

In [ ]:
rows = []
for T in THRS:
    ok = {i: (viab_keep & np.isfinite(df[sel[i]].to_numpy(np.float64)) & (df[cnt[i]].to_numpy() >= T) & (virus >= T))
          for i in (1, 2, 3)}
    rows.append({"T": int(T), "org1": int(ok[1].sum()), "org2": int(ok[2].sum()), "org3": int(ok[3].sum()),
                 "union_≥1_rep": int((ok[1] | ok[2] | ok[3]).sum()),
                 "org2∩org3": int((ok[2] & ok[3]).sum())})
div = pd.DataFrame(rows).set_index("T")
display(div)
ax = div[["org2", "org3", "union_≥1_rep", "org2∩org3"]].plot(marker="o", figsize=(8, 4.5), logy=True)
ax.set(xlabel="seuil T", ylabel="n variants", title="AAV2 -- Diversité conservée vs T")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Fit mutualisé org2 + org3 à T = `T_CHOSEN`

On empile les lignes de org2 et org3 (un variant apparaît 2× avec ses 2 mesures bruitées), F/J partagés. On fitte **λ=0** (OLS minimum-norm, poids de production) **et** la ridge à λ choisi par CV sur la grille élargie, on compare le held-out sur chaque réplicat.

In [ ]:
T_CHOSEN=5

In [ ]:
T = T_CHOSEN
i2, i3 = masks_at(T)[2], masks_at(T)[3]
y2f, y3f = df[sel[2]].to_numpy(np.float64), df[sel[3]].to_numpy(np.float64)
w2f = obs_weight(df[cnt[2]].to_numpy(np.float64), virus)
w3f = obs_weight(df[cnt[3]].to_numpy(np.float64), virus)

tr2, te2 = train_test_split(i2, test_size=0.5, random_state=0)
tr3, te3 = train_test_split(i3, test_size=0.5, random_state=0)
S_pool = np.vstack([seq_matrix[tr2], seq_matrix[tr3]])
y_pool = np.concatenate([y2f[tr2], y3f[tr3]])
w_pool = np.concatenate([w2f[tr2], w3f[tr3]])
# plus de cap 2*N_FIT sur S_pool (2026-09-18) -- solveur matrix-free, pas de plafond memoire

Fp, Jp, rankp, _ = R.fit_weights_potts_from_data_matrixfree(S_pool, y_pool, sample_weight=w_pool, verbose=False, lam=0.0)
Fp, Jp = np.asarray(Fp), np.asarray(Jp)
print(f"[§6] CV pool T={T}  (n={len(S_pool):,}, {len(LAMBDAS_CV)} λ × {KF_CV} folds)...")
Fpc, Jpc, _, infoc = R.fit_weights_potts_from_data_matrixfree(
    S_pool, y_pool, sample_weight=w_pool, lambdas_grid=LAMBDAS_CV, k_folds=KF_CV, seed=0, verbose=True, lam=None)
Fpc, Jpc = np.asarray(Fpc), np.asarray(Jpc)
lam_pool = infoc["lam"]

row_ind = sweep.loc[T]
print(f"\nT={T}  fit MUTUALISÉ org2+org3  (n={len(S_pool):,})")
for tag, F_, J_ in [("λ=0", Fp, Jp), (f"CV λ={lam_pool:.3g}", Fpc, Jpc)]:
    r2 = pearson(y2f[te2], score_FJ(seq_matrix[te2], F_, J_))
    r3 = pearson(y3f[te3], score_FJ(seq_matrix[te3], F_, J_))
    print(f"  [{tag:14s}] held-out r : org2 {r2:+.3f} | org3 {r3:+.3f}   "
          f"(fits individuels : {row_ind['r_self_org2']:+.3f} / {row_ind['r_self_org3']:+.3f})")
print(f"  plafond r(y2,y3) à T={T} : {ceil_wide.loc[T, 'org2-org3']:+.3f}")

s_pool = score_FJ(seq_matrix, Fp, Jp)
s_ind2 = score_FJ(seq_matrix, *W[(T, 2)])
s_ind3 = score_FJ(seq_matrix, *W[(T, 3)])
for k in (500, 5000, 50000):
    print(f"  top-{k:>5}  pool∩org2={topk_recovery(s_pool, s_ind2, k=k):.3f}  "
          f"pool∩org3={topk_recovery(s_pool, s_ind3, k=k):.3f}  "
          f"org2∩org3={topk_recovery(s_ind2, s_ind3, k=k):.3f}")

np.save(LIB / f"aav2_F_sel_pool_potts_sorted_readoutT{T}_unreg.npy", np.asarray(Fp, np.float32))
np.save(LIB / f"aav2_J_sel_pool_potts_sorted_readoutT{T}_unreg.npy", np.asarray(Jp, np.float32))
print(f"\n  -> lib/aav2_{{F,J}}_sel_pool_potts_sorted_readoutT{T}_unreg.npy")

## 7. Scatter — score prédit vs log2 enrichment réel (held-out, T = `T_CHOSEN`)

Le nuage `score (F + J)` vs `y = log2(organoïde_adn / virus)` sur les variants **held-out** (jamais vus au fit). Hexbin en densité log + **moyenne de `y` par bin de score ±σ** + diagonale + Pearson `r` / Spearman `ρ`. 4 panneaux : le fit mutualisé projeté sur le held-out de chaque réplicat, et chaque fit individuel sur son propre held-out (référence).

In [ ]:
def scatter_score_vs_y(ax, pred, y, title, nbin=18):
    pred, y = np.asarray(pred), np.asarray(y)
    hb = ax.hexbin(pred, y, gridsize=55, bins="log", cmap="viridis", mincnt=1)
    edges = np.quantile(pred, np.linspace(0, 1, nbin + 1))
    cx, cy, ce = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pred >= lo) & (pred <= hi)
        if m.sum() >= 30:
            cx.append(pred[m].mean()); cy.append(y[m].mean()); ce.append(y[m].std())
    ax.errorbar(cx, cy, yerr=ce, fmt="o-", color="crimson", ms=4, lw=1.3, capsize=2,
                label="moy(y) par bin de score ±σ")
    lo = min(pred.min(), y.min()); hi = max(pred.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=0.9)
    b1, b0 = np.polyfit(pred, y, 1)
    xs = np.array([pred.min(), pred.max()])
    ax.plot(xs, b0 + b1 * xs, "-", color="orange", lw=1.2, label=f"y = {b1:.2f}·score + {b0:.2f}")
    r, rho = pearson(y, pred), spearman(pred, y)
    ax.set(title=f"{title}\nr = {r:+.3f}   ρ_Spearman = {rho:+.3f}   n = {len(y):,}",
           xlabel="score prédit  (F + J)", ylabel="log2 enrichment réel  y  (held-out)")
    ax.legend(fontsize=7, loc="upper left")
    return hb


F2i, J2i = W[(T, 2)]
F3i, J3i = W[(T, 3)]
panels = [
    ("pool → org2 held-out", score_FJ(seq_matrix[te2], Fp, Jp),  y2f[te2]),
    ("pool → org3 held-out", score_FJ(seq_matrix[te3], Fp, Jp),  y3f[te3]),
    ("org2 seul → org2 held-out", score_FJ(seq_matrix[te2], F2i, J2i), y2f[te2]),
    ("org3 seul → org3 held-out", score_FJ(seq_matrix[te3], F3i, J3i), y3f[te3]),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, (ttl, p, yv) in zip(axes.ravel(), panels):
    hb = scatter_score_vs_y(ax, p, yv, ttl)
    fig.colorbar(hb, ax=ax, label="variants / cellule (log)")
fig.suptitle(f"AAV2 sélectivité — score Potts vs log2 enrichment réel  (T = {T}, held-out)", y=1.0)
plt.tight_layout(); plt.show()

## 8. Score GT du fit mutualisé sur la librairie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
finite_any = (np.isfinite(y2f) & (df[cnt[2]].to_numpy() >= T) & (virus >= T)) | \
             (np.isfinite(y3f) & (df[cnt[3]].to_numpy() >= T) & (virus >= T))
axes[0].hist(s_pool, bins=200, density=True, color="0.82", label=f"tous ({s_pool.size:,})")
axes[0].hist(s_pool[finite_any], bins=200, density=True, histtype="step", color="crimson", lw=1.3,
             label=f"mesuré à T≥{T} ({int(finite_any.sum()):,})")
axes[0].set(title=f"AAV2 -- score GT pool (F+J) — librairie de travail  (T={T})", xlabel="score", ylabel="densité")
axes[0].legend(fontsize=8)
axes[1].hexbin(s_ind2, s_pool, gridsize=50, bins="log", cmap="magma", mincnt=1)
axes[1].set(title=f"score pool vs score org2 individuel  (r={pearson(s_ind2, s_pool):+.3f})",
            xlabel="score org2 seul", ylabel="score pool org2+org3")
plt.tight_layout(); plt.show()

## 9. Conclusion

Même grille de lecture que `AAV5_SEL_potts_readout_depth.ipynb` :

- **§1** — plafond : jusqu'où un fit peut monter à chaque `T` (`r(y2,y3)` sur le jeu commun).
- **§2** — si la courbe *inter-réplicat* (bleue) et l'accord des poids (vert/olive) montent avec `T` et se rapprochent du plafond → le problème était le filtrage du readout, pas la méthode.
- **§3** — si `λ*` (grille élargie 1e-4…1e8) passe de « au bord » (T=0) à « INTÉRIEUR » (T=`T_CHOSEN`/50), la CV redevient utilisable une fois la cible débruitée.
- **§4** — F/J plus nets et mieux reproduits entre réplicats à T=`T_CHOSEN`.
- **§5** — prix à payer en nombre de variants.
- **§6/§7** — fit mutualisé org2+org3 à T=`T_CHOSEN` : held-out par réplicat vs plafond, et le scatter score vs log2 enrichment.

**⚠⚠ Chiffres ci-dessous PÉRIMÉS (2026-09-16) — calculés sur l'ancienne population avec le cap `ratio≤100`, retiré depuis (cf. markdown en tête de notebook).** À rejouer intégralement (sections 1 à 8) avant de faire confiance à quoi que ce soit ici, y compris le choix `T_CHOSEN=5` lui-même.

**Ancienne décision (2026-09-16, avant retrait du cap) : `T_CHOSEN=5`**, pas 20 (l'ancien choix par défaut repris tel quel du port AAV5). Contrairement à AAV5, `r(F2,F3)` (accord des poids entre réplicats, §2/§4) sur AAV2 n'était PAS monotone croissant en `T` — il culminait autour de `T=0-5` (+0.645/+0.647) puis **chutait** jusqu'à `T=30` (+0.185) avant de remonter à `T=100` (+0.733, mais sur seulement 6-7k variants). Ce comportement (non-monotone) pourrait changer une fois la population recalculée sans le cap ratio, puisque le cap retirait justement toute la queue haute des deux réplicats. Poids de production exportés à l'ancien `T_CHOSEN=5` : `lib/aav2_{F,J}_sel_pool_potts_sorted_readoutT5_unreg.npy` -- **également périmés**, à re-générer après avoir rejoué la sweep. (Les fichiers `..._readoutT20_unreg.npy` d'un run encore antérieur restent aussi sur disque, pas plus valides.)
